# Clasificación de Textos Históricos — Intento 3

## Cambios principales
1. **Limpieza aplicada a EVAL**: `clean_text_aggressive()` se aplica a train Y eval antes de inferencia
2. **Fix train_test_split**: `decade` se convierte a `int` antes del split; se eliminan NaN
3. **Parada manual**: callback `ManualStopCallback` — ejecuta `MANUAL_STOP['stop']=True` en otra celda para detener
4. **Anti-overfitting**: label smoothing 0.1, dropout más agresivo, L2 más fuerte, modelos más pequeños
5. **Modelos existentes conservados** (MLP, CNN, BiLSTM, BERT) — sin cambios de arquitectura
6. **Nuevos modelos**:
   - CharCNN: CNN a nivel de carácter (robusto a variantes ortográficas históricas)
   - MLP-CharNgram: TF-IDF de n-gramas de caracteres
   - Transformer Temporal: self-attention sin pesos preentrenados
   - Jerárquico (Siglo→Décadas): primero clasifica el siglo (15xx/16xx/17xx/18xx), luego la década exacta con sub-modelos especializados


In [ ]:
import subprocess
import sys
from pathlib import Path

"""
¿Que hace?:
    Instala dependencias del proyecto usando el Python del entorno activo (.venv),
    en un orden que reduce conflictos entre TensorFlow, PyTorch y Hugging Face.

Parámetros:
    Ninguno

Retorna:
    Nada
"""

def run(cmd: list[str]) -> bool:
    """
    ¿Que hace?:
        Ejecuta un comando de instalación y devuelve si terminó bien.

    Parámetros:
        cmd (list[str]): comando a ejecutar en formato de lista.

    Retorna:
        bool: True si el comando salió con código 0, False si falló.
    """
    print(f"⏳ {' '.join(cmd)}")
    result = subprocess.run(cmd)
    return result.returncode == 0


print("=" * 70)
print("📦 INSTALADOR DE DEPENDENCIAS")
print("=" * 70)
print(f"Python activo : {sys.executable}")
print(f"Versión       : {sys.version.split()[0]}")
print(f"Proyecto      : {Path.cwd()}")
print("=" * 70)

# 1) Base de empaquetado
run([sys.executable, "-m", "pip", "install", "--upgrade", "pip", "setuptools", "wheel"])

# 2) Paquetes base
BASE_PACKAGES = [
    "numpy",
    "pandas",
    "matplotlib",
    "seaborn",
    "scikit-learn",
    "tqdm",
]

# 3) ML / DL
ML_PACKAGES = [
    "tensorflow==2.15.0",
    "torch",
    "transformers",
    "datasets",
    "huggingface-hub",
    "sentencepiece",
    "accelerate",
]

print("\n🧩 Instalando paquetes base...\n")
for pkg in BASE_PACKAGES:
    ok = run([sys.executable, "-m", "pip", "install", "--upgrade", pkg])
    if not ok:
        print(f"❌ Falló: {pkg}")

print("\n🧠 Instalando paquetes de ML...\n")
for pkg in ML_PACKAGES:
    ok = run([sys.executable, "-m", "pip", "install", "--upgrade", pkg])
    if not ok:
        print(f"❌ Falló: {pkg}")

print("\n✅ Instalación finalizada.")

In [1]:
import os, sys, re, json, random, warnings, unicodedata, pickle, time
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.utils.class_weight import compute_class_weight
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import accuracy_score, f1_score, classification_report

import tensorflow as tf
import keras
from keras import layers
from keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint, Callback
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

try:
    import torch
    from torch import nn
    from torch.utils.data import Dataset, DataLoader
    from transformers import AutoTokenizer, AutoModelForSequenceClassification, get_linear_schedule_with_warmup
    BERT_AVAILABLE = True
    DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f'PyTorch: {torch.__version__} | Dispositivo: {DEVICE}')
except ImportError:
    BERT_AVAILABLE = False
    print('PyTorch/Transformers no disponibles — BERT desactivado.')

warnings.filterwarnings('ignore')
print(f'TensorFlow: {tf.__version__} | Keras: {keras.__version__}')


c:\Users\User\Documents\UNI\Sexto Semestre\Machine Learning\Proyecto\Proyecto_2-ML\.venv_ml\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


PyTorch: 2.12.0+cpu | Dispositivo: cpu
TensorFlow: 2.15.0 | Keras: 2.15.0


In [2]:
SEED = 42
random.seed(SEED); np.random.seed(SEED); tf.random.set_seed(SEED)
os.environ['PYTHONHASHSEED'] = str(SEED)

DATA_DIR  = Path('./data')
PROCESS   = Path('./process')

MAX_VOCAB           = 80000
MAX_LEN             = 400
EMBED_DIM           = 200
CHAR_MAX_LEN        = 1000
CHAR_EMBED_DIM      = 64
BATCH_SIZE          = 32
NUM_EPOCHS          = 100
MIN_TEXT_LEN        = 20
MIN_SAMPLES_CLASS   = 600
LABEL_SMOOTHING     = 0.1

for sub in ['keras','pkl','images','bert']:
    (PROCESS / sub).mkdir(parents=True, exist_ok=True)

kpath = lambda n: str(PROCESS/'keras'/n)
ppath = lambda n: str(PROCESS/'pkl'/n)
ipath = lambda n: str(PROCESS/'images'/n)

print('Configuracion lista.')


Configuracion lista.


## 1. Carga de datos

In [3]:
df_train = pd.read_csv(DATA_DIR / 'train.csv')
df_eval  = pd.read_csv(DATA_DIR / 'eval.csv')

print(f'Train: {df_train.shape}')
print(f'Eval : {df_eval.shape}')
print(f'Columnas train: {df_train.columns.tolist()}')
print(f'Columnas eval : {df_eval.columns.tolist()}')
print(f'Tipo decade   : {df_train["decade"].dtype}')
print(f'Decadas unicas: {sorted(df_train["decade"].unique())}')


Train: (31403, 2)
Eval : (3490, 2)
Columnas train: ['text', 'decade']
Columnas eval : ['id', 'text']
Tipo decade   : int64
Decadas unicas: [150, 151, 152, 153, 154, 155, 156, 157, 158, 159, 160, 161, 162, 163, 164, 165, 166, 167, 168, 169, 170, 171, 172, 173, 174, 175, 176, 177, 178, 179, 180, 181, 182, 183, 184, 185, 186, 187, 188]


## 2. Limpieza de texto

**La misma función se aplica a train y eval.** El modelo fue entrenado con texto limpio,
por lo que usar texto sucio en inferencia degradaría las predicciones.


In [4]:
def clean_text_aggressive(text: str) -> str:
    '''Limpieza agresiva para textos historicos con OCR corrupto.'''
    if not isinstance(text, str):
        text = str(text)

    # 1. Normalizacion Unicode NFC
    text = unicodedata.normalize('NFC', text)

    # 2. Caracteres Unicode corruptos
    text = text.replace('\ufffd', ' ').replace('\xa0', ' ')

    # 3. Reunir palabras divididas por guion al final de linea
    text = re.sub(r'(?<=\w)[\-\u00ad\u00ac\u00b7\u2022]\s*(?=\w)', '', text)

    # 4. Metadata de bibliotecas digitales
    metadata_kw = r'(?i)(copyright|google|proquest|biblioteca\s+nacional|digitized\s+by|'\
                  r'archive\.org|hathitrust|british\s+library|internet\s+archive|'\
                  r'university\s+library|gallica|europeana)[^\n]*\n?'
    text = re.sub(metadata_kw, ' ', text)

    # 5. Numeracion de paginas
    text = re.sub(r'[-\u2014\u2013]\s*\d{1,4}\s*[-\u2014\u2013]', ' ', text)
    text = re.sub(r'(?i)\b(p[a\u00e1]g|fol|p)\.\s*\d+', ' ', text)

    # 6. Filtrar lineas corruptas
    clean_lines = []
    for line in text.split('\n'):
        letters = sum(c.isalpha() for c in line)
        digits  = sum(c.isdigit() for c in line)
        puncts  = sum(not c.isalnum() and not c.isspace() for c in line)
        sym_runs= len(re.findall(r'[^\w\s]{2,}', line))
        if letters < 3 and (digits + puncts) > 3: continue
        if puncts >= 8 and letters <= 6:          continue
        if sym_runs >= 2:                         continue
        if re.search(r'\d[\s,.\-]\d[\s,.\-]\d', line): continue
        clean_lines.append(line)
    text = '\n'.join(clean_lines)

    # 7. Caracteres de control
    text = re.sub(r'[\x00-\x08\x0e-\x1f\x7f]', ' ', text)

    # 8. Normalizar saltos de linea
    text = re.sub(r'[\r\n]+', ' ', text)

    # 9. Simbolos aislados
    text = re.sub(r'(?<!\w)[#%@\\|~`<>{}\[\]\*](?!\w)', ' ', text)

    # 10. Simbolos repetidos
    text = re.sub(r'([^\w\s])\1{2,}', r'\1', text)

    # 11. Numeros de 1-2 digitos aislados (ruido OCR)
    text = re.sub(r'(?<!\w)\d{1,2}(?!\w)', ' ', text)

    # 12. Espacios multiples
    text = re.sub(r' {2,}', ' ', text).strip()

    return text

print('clean_text_aggressive definida.')


clean_text_aggressive definida.


In [5]:
print('Limpiando train...')
df_train['text_clean'] = df_train['text'].apply(clean_text_aggressive)

print('Limpiando eval (misma funcion)...')
df_eval['text_clean']  = df_eval['text'].apply(clean_text_aggressive)

# Filtrar train con texto demasiado corto; mantener TODOS los de eval
mask = df_train['text_clean'].str.len() >= MIN_TEXT_LEN
print(f'Train: {mask.sum()}/{len(df_train)} filas conservadas')
df_train = df_train[mask].reset_index(drop=True)

short_eval = (df_eval['text_clean'].str.len() < MIN_TEXT_LEN).sum()
if short_eval:
    print(f'Eval: {short_eval} filas con texto muy corto — se conservan para prediccion')

print(f'Train final: {df_train.shape} | Eval final: {df_eval.shape}')


Limpiando train...
Limpiando eval (misma funcion)...
Train: 31224/31403 filas conservadas
Eval: 22 filas con texto muy corto — se conservan para prediccion
Train final: (31224, 3) | Eval final: (3490, 3)


## 3. EDA

In [6]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

decade_counts = df_train['decade'].value_counts().sort_index()
axes[0].bar(range(len(decade_counts)), decade_counts.values, color='steelblue', alpha=0.8)
axes[0].set_xticks(range(len(decade_counts)))
axes[0].set_xticklabels([str(d) for d in decade_counts.index], rotation=90, fontsize=7)
axes[0].set_title('Distribucion de Decadas'); axes[0].set_xlabel('Decada'); axes[0].set_ylabel('Muestras')

lens = df_train['text_clean'].str.len()
axes[1].hist(lens, bins=50, color='steelblue', alpha=0.8)
axes[1].axvline(lens.mean(), color='red', linestyle='--', label=f'Media: {lens.mean():.0f}')
axes[1].set_title('Longitud de textos (post-limpieza)'); axes[1].legend()

plt.tight_layout()
plt.savefig(ipath('eda_distribucion.png'), dpi=100, bbox_inches='tight')
plt.show()
print(f'Decadas: {df_train["decade"].nunique()} | Media longitud: {lens.mean():.0f} chars')


Decadas: 39 | Media longitud: 458 chars


## 4. Codificacion de labels

**Fix**: `decade` se convierte explicitamente a `int` antes de `LabelEncoder` y `train_test_split`.
Esto resuelve el fallo por datos no-numericos.


In [26]:
# FIX: Eliminar NaN y forzar tipo entero
df_train = df_train.dropna(subset=['decade']).reset_index(drop=True)
df_train['decade'] = pd.to_numeric(df_train['decade'], errors='coerce').dropna().astype(int)
df_train = df_train.dropna(subset=['decade']).reset_index(drop=True)
df_train['decade'] = df_train['decade'].astype(int)

print(f'Tipo decade: {df_train["decade"].dtype}')
print(f'Rango: {df_train["decade"].min()} - {df_train["decade"].max()}')

label_encoder = LabelEncoder()
label_encoder.fit(df_train['decade'])
NUM_CLASSES = len(label_encoder.classes_)
print(f'Clases: {NUM_CLASSES} | Decadas: {label_encoder.classes_}')

with open(ppath('label_encoder.pkl'), 'wb') as f:
    pickle.dump(label_encoder, f)


Tipo decade: int32
Rango: 150 - 188
Clases: 39 | Decadas: [150 151 152 153 154 155 156 157 158 159 160 161 162 163 164 165 166 167
 168 169 170 171 172 173 174 175 176 177 178 179 180 181 182 183 184 185
 186 187 188]


## 5. Augmentacion de texto

In [27]:
def augment_text_historical(text: str, p: float = 0.3) -> str:
    '''Variantes ortograficas historicas del espanol (s.XV-XIX).'''
    result = []
    for c in text:
        r = random.random()
        if c.islower():
            if c in 'uv' and r < p:
                result.append('v' if c == 'u' else 'u')
            elif c in 'iy' and r < p * 0.5:
                result.append('y' if c == 'i' else 'i')
            else:
                result.append(c)
        else:
            result.append(c)
    return ''.join(result)

# Augmentar clases con pocas muestras
df_aug = df_train.copy()
for decade_val in df_train['decade'].unique():
    subset = df_train[df_train['decade'] == decade_val]
    n_need = max(0, MIN_SAMPLES_CLASS - len(subset))
    if n_need > 0:
        extra = subset.sample(n=n_need, replace=True, random_state=SEED).copy()
        extra['text_clean'] = extra['text_clean'].apply(lambda t: augment_text_historical(t, 0.3))
        df_aug = pd.concat([df_aug, extra], ignore_index=True)

df_aug['label'] = label_encoder.transform(df_aug['decade'].astype(int))
print(f'Original: {len(df_train)} | Augmentado: {len(df_aug)}')
print(f'Min por clase: {df_aug["label"].value_counts().min()}')


Original: 31224 | Augmentado: 31224
Min por clase: 750


## 6. Train / Val / Test split

**Fix clave**: `y_all` ya contiene enteros (labels codificados) — esto resuelve el fallo de `train_test_split`.


In [28]:
X_all = df_aug['text_clean'].astype(str).to_numpy(dtype=object)
y_all = df_aug['label'].values  # enteros: 0..NUM_CLASSES-1

# Verificacion de integridad
assert np.issubdtype(y_all.dtype, np.integer), f'y_all debe ser entero, es {y_all.dtype}'
assert not np.any(pd.isna(y_all)), 'y_all contiene NaN'

X_train, X_temp, y_train, y_temp = train_test_split(
    X_all, y_all, test_size=0.20, random_state=SEED, stratify=y_all)

X_val, X_test_int, y_val, y_test_int = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=SEED, stratify=y_temp)

print(f'Train: {len(X_train)} | Val: {len(X_val)} | Test interno: {len(X_test_int)}')
print(f'Clases en train: {len(np.unique(y_train))} | en val: {len(np.unique(y_val))}')

# Pesos de clases
cw_arr  = compute_class_weight('balanced', classes=np.arange(NUM_CLASSES), y=y_train)
cw_dict = {i: w for i, w in enumerate(cw_arr)}
print(f'Pesos (min/max): {cw_arr.min():.3f} / {cw_arr.max():.3f}')

# One-hot para label smoothing
y_train_oh   = tf.keras.utils.to_categorical(y_train,    NUM_CLASSES)
y_val_oh     = tf.keras.utils.to_categorical(y_val,      NUM_CLASSES)
y_test_int_oh= tf.keras.utils.to_categorical(y_test_int, NUM_CLASSES)

# Labels de siglo para modelo jerarquico
def encoded_to_century(enc):
    decade = int(label_encoder.inverse_transform([enc])[0])
    return {15:0, 16:1, 17:2, 18:3}.get(decade // 10, 0)

century_train = np.array([encoded_to_century(y) for y in y_train])
century_val   = np.array([encoded_to_century(y) for y in y_val])
century_test  = np.array([encoded_to_century(y) for y in y_test_int])
print(f'Siglos en train: {np.unique(century_train)}')


Train: 24979 | Val: 3122 | Test interno: 3123
Clases en train: 39 | en val: 39
Pesos (min/max): 0.955 / 1.067
Siglos en train: [0 1 2 3]


## 7. Callback de parada manual

```python
# Para detener cualquier modelo en la siguiente epoca:
MANUAL_STOP['stop'] = True
```

El flag se resetea automaticamente. No es necesario interrumpir el kernel.


In [ ]:
MANUAL_STOP = {'stop': False}

class ManualStopCallback(Callback):
    '''Detiene entrenamiento cuando MANUAL_STOP["stop"] == True.'''
    def on_epoch_end(self, epoch, logs=None):
        if MANUAL_STOP.get('stop', False):
            print(f'\n[PARADA MANUAL] Detenido en epoca {epoch+1}. Pesos del mejor epoch restaurados por EarlyStopping.')
            self.model.stop_training = True
            MANUAL_STOP['stop'] = False

# Celda de parada — ejecutar mientras entrena:
# MANUAL_STOP['stop'] = True

print('ManualStopCallback listo.')
print('Para detener: ejecuta  MANUAL_STOP["stop"] = True  en otra celda.')


## 8. Extraccion de features

In [ ]:
# --- TF-IDF palabras (word n-grams) ---
print('TF-IDF de palabras...')
tfidf_word = TfidfVectorizer(
    max_features=80000, ngram_range=(1,3), analyzer='word',
    sublinear_tf=True, min_df=3, max_df=0.95,
    strip_accents=None, token_pattern=r'(?u)\b[\w\u00C0-\u024F]{2,}\b',
)
X_train_tfidf = tfidf_word.fit_transform(X_train).toarray().astype(np.float32)
X_val_tfidf   = tfidf_word.transform(X_val).toarray().astype(np.float32)
X_test_tfidf  = tfidf_word.transform(X_test_int).toarray().astype(np.float32)
X_eval_tfidf  = tfidf_word.transform(df_eval['text_clean'].astype(str)).toarray().astype(np.float32)
with open(ppath('tfidf_word.pkl'), 'wb') as f: pickle.dump(tfidf_word, f)
print(f'Word TF-IDF: {X_train_tfidf.shape}')

# --- TF-IDF caracteres (char n-grams) — perspectiva diferente ---
print('TF-IDF de caracteres...')
tfidf_char = TfidfVectorizer(
    max_features=50000, ngram_range=(2,5), analyzer='char_wb',
    sublinear_tf=True, min_df=3, max_df=0.95,
)
X_train_ctfidf = tfidf_char.fit_transform(X_train).toarray().astype(np.float32)
X_val_ctfidf   = tfidf_char.transform(X_val).toarray().astype(np.float32)
X_test_ctfidf  = tfidf_char.transform(X_test_int).toarray().astype(np.float32)
X_eval_ctfidf  = tfidf_char.transform(df_eval['text_clean'].astype(str)).toarray().astype(np.float32)
with open(ppath('tfidf_char.pkl'), 'wb') as f: pickle.dump(tfidf_char, f)
print(f'Char TF-IDF: {X_train_ctfidf.shape}')

# --- Secuencias de palabras ---
print('Secuencias de palabras...')
tokenizer_seq = Tokenizer(num_words=MAX_VOCAB, oov_token='<OOV>')
tokenizer_seq.fit_on_texts(X_train)
to_seq = lambda txts: pad_sequences(tokenizer_seq.texts_to_sequences(txts), maxlen=MAX_LEN, padding='post', truncating='post')
X_train_seq = to_seq(X_train); X_val_seq = to_seq(X_val)
X_test_seq  = to_seq(X_test_int)
X_eval_seq  = to_seq(df_eval['text_clean'].astype(str).values)
VOCAB_SIZE   = min(len(tokenizer_seq.word_index)+1, MAX_VOCAB)
with open(ppath('tokenizer_seq.pkl'), 'wb') as f: pickle.dump(tokenizer_seq, f)
print(f'Seq palabras: {X_train_seq.shape} | vocab: {VOCAB_SIZE}')

# --- Secuencias de caracteres ---
print('Secuencias de caracteres...')
CHAR_CHARS  = 'abcdefghijklmnopqrstuvwxyz\u00e1\u00e9\u00ed\u00f3\u00fa\u00fc\u00f1\u00e0\u00e8\u00ec\u00f2\u00f9\u00e2\u00ea\u00ee\u00f4\u00fb\u00e4\u00eb\u00ef\u00f6\u00fc0123456789 .,;:!?-\'\n'
CHAR2IDX    = {c: i+2 for i, c in enumerate(CHAR_CHARS)}
CHAR_VOCAB  = len(CHAR2IDX)+2
def txt2char(text, ml=CHAR_MAX_LEN):
    s = [CHAR2IDX.get(c.lower(), 1) for c in str(text)[:ml]]
    return s + [0]*max(0, ml-len(s))
X_train_char = np.array([txt2char(t) for t in X_train], dtype=np.int32)
X_val_char   = np.array([txt2char(t) for t in X_val],   dtype=np.int32)
X_test_char  = np.array([txt2char(t) for t in X_test_int], dtype=np.int32)
X_eval_char  = np.array([txt2char(t) for t in df_eval['text_clean'].astype(str)], dtype=np.int32)
print(f'Seq caracteres: {X_train_char.shape} | char_vocab: {CHAR_VOCAB}')


## 9. Modelos existentes (arquitectura original + ManualStop + label smoothing)

Los modelos MLP, CNN y BiLSTM conservan la misma arquitectura del intento 2.
Se agrega `ManualStopCallback` y se usa label smoothing en el loss.


In [ ]:
# Helpers de callbacks comunes
def get_callbacks(model_name, monitor='val_accuracy', patience=12):
    return [
        EarlyStopping(monitor=monitor, patience=patience, restore_best_weights=True, verbose=1),
        ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=5, min_lr=1e-7, verbose=1),
        ModelCheckpoint(kpath(f'best_{model_name}.keras'), monitor=monitor, save_best_only=True, verbose=0),
        ManualStopCallback(),
    ]

smooth_loss = keras.losses.CategoricalCrossentropy(label_smoothing=LABEL_SMOOTHING)


### Modelo 1: MLP v2 (original) + ManualStop

In [ ]:
def build_mlp_v2(input_dim, num_classes):
    inputs = keras.Input(shape=(input_dim,), name='tfidf_input')
    x = layers.Dense(2048, kernel_regularizer=keras.regularizers.l2(1e-4))(inputs)
    x = layers.BatchNormalization()(x); x = layers.Activation('relu')(x); x = layers.Dropout(0.5)(x)
    x = layers.Dense(1024, kernel_regularizer=keras.regularizers.l2(1e-4))(x)
    x = layers.BatchNormalization()(x); x = layers.Activation('relu')(x); x = layers.Dropout(0.4)(x)
    x = layers.Dense(512,  kernel_regularizer=keras.regularizers.l2(1e-4))(x)
    x = layers.BatchNormalization()(x); x = layers.Activation('relu')(x); x = layers.Dropout(0.3)(x)
    x = layers.Dense(256, activation='relu')(x); x = layers.Dropout(0.2)(x)
    out = layers.Dense(num_classes, activation='softmax')(x)
    return keras.Model(inputs, out, name='MLP_v2')

model_mlp = build_mlp_v2(X_train_tfidf.shape[1], NUM_CLASSES)
model_mlp.compile(optimizer=keras.optimizers.Adam(1e-3), loss=smooth_loss, metrics=['accuracy'])
model_mlp.summary()


In [ ]:
history_mlp = model_mlp.fit(
    X_train_tfidf, y_train_oh,
    validation_data=(X_val_tfidf, y_val_oh),
    epochs=NUM_EPOCHS, batch_size=BATCH_SIZE,
    class_weight=cw_dict, callbacks=get_callbacks('mlp_v2'), verbose=1)
model_mlp.save(kpath('modelo_mlp_v2_final.keras'))

mlp_val  = accuracy_score(y_val,     np.argmax(model_mlp.predict(X_val_tfidf,  verbose=0), 1))
mlp_test = accuracy_score(y_test_int, np.argmax(model_mlp.predict(X_test_tfidf, verbose=0), 1))
print(f'MLP v2  — Val: {mlp_val:.4f} | Test: {mlp_test:.4f}')


### Modelo 2: CNN v2 (original) + ManualStop

In [ ]:
def build_cnn_v2(vocab_size, embed_dim, max_len, num_classes):
    inputs = keras.Input(shape=(max_len,), name='seq_input')
    emb = layers.Embedding(vocab_size, embed_dim)(inputs)
    emb = layers.SpatialDropout1D(0.3)(emb)
    branches = []
    for ks in [2, 3, 4, 5]:
        c = layers.Conv1D(512, ks, activation='relu', padding='same')(emb)
        branches.append(layers.GlobalMaxPooling1D()(c))
    x = layers.Concatenate()(branches)
    x = layers.Dense(512, activation='relu')(x); x = layers.BatchNormalization()(x); x = layers.Dropout(0.5)(x)
    x = layers.Dense(256, activation='relu')(x); x = layers.Dropout(0.3)(x)
    out = layers.Dense(num_classes, activation='softmax')(x)
    return keras.Model(inputs, out, name='CNN_v2')

model_cnn = build_cnn_v2(VOCAB_SIZE, EMBED_DIM, MAX_LEN, NUM_CLASSES)
model_cnn.compile(optimizer=keras.optimizers.Adam(1e-3), loss=smooth_loss, metrics=['accuracy'])
model_cnn.summary()


In [ ]:
history_cnn = model_cnn.fit(
    X_train_seq, y_train_oh,
    validation_data=(X_val_seq, y_val_oh),
    epochs=NUM_EPOCHS, batch_size=BATCH_SIZE,
    class_weight=cw_dict, callbacks=get_callbacks('cnn_v2'), verbose=1)
model_cnn.save(kpath('modelo_cnn_v2_final.keras'))

cnn_val  = accuracy_score(y_val,      np.argmax(model_cnn.predict(X_val_seq,  verbose=0), 1))
cnn_test = accuracy_score(y_test_int, np.argmax(model_cnn.predict(X_test_seq, verbose=0), 1))
print(f'CNN v2  — Val: {cnn_val:.4f} | Test: {cnn_test:.4f}')


### Modelo 3: BiLSTM v2 (original) + ManualStop

In [ ]:
class BahdanauAttention(layers.Layer):
    def __init__(self, units=64, **kwargs):
        super().__init__(**kwargs)
        self.units = units
        self.W = layers.Dense(units, use_bias=False)
        self.V = layers.Dense(1, use_bias=False)
    def call(self, hs, training=None):
        score = self.V(tf.nn.tanh(self.W(hs)))
        attn  = tf.nn.softmax(score, axis=1)
        return tf.reduce_sum(hs * attn, axis=1)
    def get_config(self):
        cfg = super().get_config(); cfg['units'] = self.units; return cfg

def build_bilstm_v2(vocab_size, embed_dim, max_len, num_classes):
    inputs = keras.Input(shape=(max_len,))
    x = layers.Embedding(vocab_size, embed_dim)(inputs)
    x = layers.SpatialDropout1D(0.3)(x)
    x = layers.Bidirectional(layers.LSTM(256, return_sequences=True, dropout=0.2, recurrent_dropout=0.1))(x)
    x = layers.Bidirectional(layers.LSTM(128, return_sequences=True, dropout=0.2, recurrent_dropout=0.1))(x)
    x = BahdanauAttention(units=64, name='attention')(x)
    x = layers.Dense(512, activation='relu')(x); x = layers.BatchNormalization()(x); x = layers.Dropout(0.5)(x)
    x = layers.Dense(256, activation='relu')(x); x = layers.Dropout(0.3)(x)
    out = layers.Dense(num_classes, activation='softmax')(x)
    return keras.Model(inputs, out, name='BiLSTM_v2')

model_bilstm = build_bilstm_v2(VOCAB_SIZE, EMBED_DIM, MAX_LEN, NUM_CLASSES)
model_bilstm.compile(optimizer=keras.optimizers.Adam(5e-4, clipnorm=1.0), loss=smooth_loss, metrics=['accuracy'])
model_bilstm.summary()


In [ ]:
history_bilstm = model_bilstm.fit(
    X_train_seq, y_train_oh,
    validation_data=(X_val_seq, y_val_oh),
    epochs=NUM_EPOCHS, batch_size=BATCH_SIZE,
    class_weight=cw_dict, callbacks=get_callbacks('bilstm_v2'), verbose=1)
model_bilstm.save(kpath('modelo_bilstm_v2_final.keras'))

bilstm_val  = accuracy_score(y_val,      np.argmax(model_bilstm.predict(X_val_seq,  verbose=0), 1))
bilstm_test = accuracy_score(y_test_int, np.argmax(model_bilstm.predict(X_test_seq, verbose=0), 1))
print(f'BiLSTM  — Val: {bilstm_val:.4f} | Test: {bilstm_test:.4f}')


## 10. Nuevos modelos

Cada modelo aborda el problema desde una perspectiva diferente.


### Modelo 4 (NUEVO): CharCNN — CNN a nivel de caracter

Opera sobre caracteres individuales en lugar de palabras. Robusto a las variantes ortograficas
historicas (u/v, i/y, variaciones de tildes) ya que aprende patrones morfologicos directamente.


In [ ]:
def build_char_cnn(char_vocab_size, embed_dim, max_len, num_classes):
    '''CNN multi-kernel sobre secuencias de caracteres.'''
    inputs = keras.Input(shape=(max_len,), name='char_input')
    x = layers.Embedding(char_vocab_size, embed_dim)(inputs)
    x = layers.SpatialDropout1D(0.2)(x)

    branches = []
    for ks in [3, 5, 7, 11]:  # kernels grandes = patrones morfologicos mas largos
        c = layers.Conv1D(256, ks, activation='relu', padding='same',
                          kernel_regularizer=keras.regularizers.l2(1e-4))(x)
        c = layers.MaxPooling1D(3, padding='same')(c)
        c = layers.Conv1D(256, ks, activation='relu', padding='same')(c)
        branches.append(layers.GlobalMaxPooling1D()(c))

    x = layers.Concatenate()(branches)  # 4 x 256 = 1024
    x = layers.Dense(512, activation='relu', kernel_regularizer=keras.regularizers.l2(1e-4))(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(0.5)(x)
    x = layers.Dense(256, activation='relu')(x)
    x = layers.Dropout(0.4)(x)
    out = layers.Dense(num_classes, activation='softmax')(x)
    return keras.Model(inputs, out, name='CharCNN')

model_charcnn = build_char_cnn(CHAR_VOCAB, CHAR_EMBED_DIM, CHAR_MAX_LEN, NUM_CLASSES)
model_charcnn.compile(optimizer=keras.optimizers.Adam(1e-3), loss=smooth_loss, metrics=['accuracy'])
model_charcnn.summary()


In [ ]:
history_charcnn = model_charcnn.fit(
    X_train_char, y_train_oh,
    validation_data=(X_val_char, y_val_oh),
    epochs=NUM_EPOCHS, batch_size=BATCH_SIZE,
    class_weight=cw_dict, callbacks=get_callbacks('charcnn'), verbose=1)
model_charcnn.save(kpath('modelo_charcnn_final.keras'))

charcnn_val  = accuracy_score(y_val,      np.argmax(model_charcnn.predict(X_val_char,  verbose=0), 1))
charcnn_test = accuracy_score(y_test_int, np.argmax(model_charcnn.predict(X_test_char, verbose=0), 1))
print(f'CharCNN — Val: {charcnn_val:.4f} | Test: {charcnn_test:.4f}')


### Modelo 5 (NUEVO): MLP con TF-IDF de char n-gramas

Usa n-gramas de caracteres (2-5 chars) como features en lugar de palabras completas.
Captura patrones morfologicos sin depender de segmentacion por palabras.


In [ ]:
def build_mlp_char_ngram(input_dim, num_classes):
    inputs = keras.Input(shape=(input_dim,), name='char_tfidf_input')
    x = layers.Dense(1024, kernel_regularizer=keras.regularizers.l2(5e-4))(inputs)
    x = layers.BatchNormalization()(x); x = layers.Activation('relu')(x); x = layers.Dropout(0.55)(x)
    x = layers.Dense(512, kernel_regularizer=keras.regularizers.l2(5e-4))(x)
    x = layers.BatchNormalization()(x); x = layers.Activation('relu')(x); x = layers.Dropout(0.45)(x)
    x = layers.Dense(256, activation='relu')(x); x = layers.Dropout(0.35)(x)
    out = layers.Dense(num_classes, activation='softmax')(x)
    return keras.Model(inputs, out, name='MLP_CharNgram')

model_mlpchar = build_mlp_char_ngram(X_train_ctfidf.shape[1], NUM_CLASSES)
model_mlpchar.compile(optimizer=keras.optimizers.Adam(5e-4), loss=smooth_loss, metrics=['accuracy'])
model_mlpchar.summary()


In [ ]:
history_mlpchar = model_mlpchar.fit(
    X_train_ctfidf, y_train_oh,
    validation_data=(X_val_ctfidf, y_val_oh),
    epochs=NUM_EPOCHS, batch_size=BATCH_SIZE,
    class_weight=cw_dict, callbacks=get_callbacks('mlp_charngram'), verbose=1)
model_mlpchar.save(kpath('modelo_mlp_charngram_final.keras'))

mlpchar_val  = accuracy_score(y_val,      np.argmax(model_mlpchar.predict(X_val_ctfidf,  verbose=0), 1))
mlpchar_test = accuracy_score(y_test_int, np.argmax(model_mlpchar.predict(X_test_ctfidf, verbose=0), 1))
print(f'MLP-CharNgram — Val: {mlpchar_val:.4f} | Test: {mlpchar_test:.4f}')


### Modelo 6 (NUEVO): Transformer Temporal

Self-attention sin pesos preentrenados. Aprende dependencias de largo alcance en el texto.
Positional encoding aprendido (no sinusoidal fijo) para adaptarse mejor al dominio.


In [ ]:
class TransformerBlock(layers.Layer):
    def __init__(self, embed_dim, num_heads, ff_dim, dropout=0.15, **kwargs):
        super().__init__(**kwargs)
        self.embed_dim = embed_dim; self.num_heads = num_heads
        self.ff_dim = ff_dim; self.dropout_rate = dropout
        self.att   = layers.MultiHeadAttention(num_heads=num_heads, key_dim=embed_dim//num_heads)
        self.ffn   = keras.Sequential([
            layers.Dense(ff_dim, activation='gelu'),
            layers.Dropout(dropout),
            layers.Dense(embed_dim),
        ])
        self.norm1 = layers.LayerNormalization(epsilon=1e-6)
        self.norm2 = layers.LayerNormalization(epsilon=1e-6)
        self.drop1 = layers.Dropout(dropout)
        self.drop2 = layers.Dropout(dropout)

    def call(self, x, training=None):
        attn = self.drop1(self.att(x, x, training=training), training=training)
        x    = self.norm1(x + attn)
        ffn  = self.drop2(self.ffn(x), training=training)
        return self.norm2(x + ffn)

    def get_config(self):
        cfg = super().get_config()
        cfg.update({'embed_dim':self.embed_dim,'num_heads':self.num_heads,
                    'ff_dim':self.ff_dim,'dropout':self.dropout_rate})
        return cfg

def build_temporal_transformer(vocab_size, embed_dim, max_len, num_classes,
                                num_heads=4, ff_dim=256, num_blocks=2):
    '''Transformer con positional embedding aprendido.'''
    inputs  = keras.Input(shape=(max_len,))
    tok_emb = layers.Embedding(vocab_size, embed_dim)(inputs)
    pos_emb = layers.Embedding(max_len,    embed_dim)(tf.range(tf.shape(inputs)[1]))
    x = tok_emb + pos_emb
    x = layers.SpatialDropout1D(0.2)(x)
    for _ in range(num_blocks):
        x = TransformerBlock(embed_dim, num_heads, ff_dim, dropout=0.15)(x)
    x = layers.GlobalAveragePooling1D()(x)
    x = layers.Dense(256, activation='gelu', kernel_regularizer=keras.regularizers.l2(1e-4))(x)
    x = layers.Dropout(0.4)(x)
    x = layers.Dense(128, activation='gelu')(x)
    x = layers.Dropout(0.3)(x)
    out = layers.Dense(num_classes, activation='softmax')(x)
    return keras.Model(inputs, out, name='TemporalTransformer')

model_transf = build_temporal_transformer(VOCAB_SIZE, EMBED_DIM, MAX_LEN, NUM_CLASSES)
model_transf.compile(optimizer=keras.optimizers.Adam(3e-4, clipnorm=1.0), loss=smooth_loss, metrics=['accuracy'])
model_transf.summary()


In [ ]:
history_transf = model_transf.fit(
    X_train_seq, y_train_oh,
    validation_data=(X_val_seq, y_val_oh),
    epochs=NUM_EPOCHS, batch_size=BATCH_SIZE,
    class_weight=cw_dict, callbacks=get_callbacks('transformer'), verbose=1)
model_transf.save(kpath('modelo_transformer_final.keras'))

transf_val  = accuracy_score(y_val,      np.argmax(model_transf.predict(X_val_seq,  verbose=0), 1))
transf_test = accuracy_score(y_test_int, np.argmax(model_transf.predict(X_test_seq, verbose=0), 1))
print(f'Transformer — Val: {transf_val:.4f} | Test: {transf_test:.4f}')


### Modelo 7 (NUEVO): Jerarquico Siglo → Decada

**Aproximacion de dos etapas:**
1. **Clasificador de siglo**: modelo ligero que predice el siglo (15xx / 16xx / 17xx / 18xx)
2. **Sub-modelos por siglo**: un modelo especializado por cada siglo que clasifica solo las decadas de ese siglo

La idea es que cada sub-modelo aprende los patrones linguisticos *especificos* de su siglo, sin contaminacion
de otros periodos historicos. El clasificador de siglo actua como router.


In [ ]:
class HierarchicalModel:
    '''Clasificador jerarquico Siglo -> Decada.'''

    def __init__(self):
        self.century_model   = None
        self.decade_submodels= {}   # {century_idx: keras.Model}
        self.century_les     = {}   # {century_idx: LabelEncoder local}

    @staticmethod
    def _build_century_clf(input_dim, n_centuries):
        inp = keras.Input(shape=(input_dim,))
        x = layers.Dense(512, kernel_regularizer=keras.regularizers.l2(5e-4))(inp)
        x = layers.BatchNormalization()(x); x = layers.Activation('relu')(x); x = layers.Dropout(0.5)(x)
        x = layers.Dense(256, activation='relu')(x); x = layers.Dropout(0.4)(x)
        out = layers.Dense(n_centuries, activation='softmax')(x)
        m = keras.Model(inp, out, name='century_clf')
        m.compile(optimizer=keras.optimizers.Adam(1e-3),
                  loss=keras.losses.SparseCategoricalCrossentropy(), metrics=['accuracy'])
        return m

    @staticmethod
    def _build_decade_clf(input_dim, n_decades, century_idx):
        inp = keras.Input(shape=(input_dim,))
        x = layers.Dense(256, kernel_regularizer=keras.regularizers.l2(5e-4))(inp)
        x = layers.BatchNormalization()(x); x = layers.Activation('relu')(x); x = layers.Dropout(0.5)(x)
        x = layers.Dense(128, activation='relu')(x); x = layers.Dropout(0.4)(x)
        out = layers.Dense(n_decades, activation='softmax')(x)
        m = keras.Model(inp, out, name=f'decade_clf_c{century_idx}')
        m.compile(optimizer=keras.optimizers.Adam(1e-3),
                  loss=keras.losses.SparseCategoricalCrossentropy(), metrics=['accuracy'])
        return m

    def fit(self, X_tr, y_tr, X_v, y_v, cent_tr, cent_v,
            batch_size=32, epochs=80):
        n_centuries = len(np.unique(cent_tr))

        # Etapa 1: clasificador de siglo
        print('=== Etapa 1: Clasificador de Siglo ===')
        self.century_model = self._build_century_clf(X_tr.shape[1], n_centuries)
        cb_cent = [
            EarlyStopping('val_accuracy', patience=10, restore_best_weights=True, verbose=1),
            ReduceLROnPlateau('val_loss', factor=0.5, patience=5, min_lr=1e-7, verbose=1),
            ManualStopCallback(),
        ]
        self.century_model.fit(X_tr, cent_tr, validation_data=(X_v, cent_v),
                               epochs=epochs, batch_size=batch_size,
                               callbacks=cb_cent, verbose=1)

        # Etapa 2: sub-modelos por siglo
        for c_idx in np.unique(cent_tr):
            mask_tr = cent_tr == c_idx
            mask_v  = cent_v  == c_idx
            X_tr_c  = X_tr[mask_tr]; y_tr_c = y_tr[mask_tr]
            X_v_c   = X_v[mask_v];   y_v_c  = y_v[mask_v]

            if len(X_tr_c) < 20:
                print(f'  Siglo {c_idx+15}xx: solo {len(X_tr_c)} muestras, skip.')
                continue

            print(f'\n=== Etapa 2: Sub-modelo Siglo {c_idx+15}xx ({len(X_tr_c)} muestras) ===')
            le_local = LabelEncoder()
            y_tr_local = le_local.fit_transform(y_tr_c)
            y_v_local  = le_local.transform(y_v_c) if len(y_v_c) > 0 else np.array([])
            self.century_les[c_idx] = le_local

            n_dec = len(le_local.classes_)
            sub   = self._build_decade_clf(X_tr_c.shape[1], n_dec, c_idx)
            cb_sub= [
                EarlyStopping('val_accuracy' if len(y_v_local)>0 else 'accuracy',
                              patience=8, restore_best_weights=True, verbose=1),
                ReduceLROnPlateau('val_loss' if len(y_v_local)>0 else 'loss',
                                  factor=0.5, patience=4, min_lr=1e-6, verbose=1),
                ManualStopCallback(),
            ]
            val_data = (X_v_c, y_v_local) if len(y_v_local)>0 else None
            sub.fit(X_tr_c, y_tr_local, validation_data=val_data,
                    epochs=epochs, batch_size=min(batch_size, len(X_tr_c)),
                    callbacks=cb_sub, verbose=1)
            self.decade_submodels[c_idx] = sub
            print(f'  Sub-modelo siglo {c_idx+15}xx: {n_dec} decadas aprendidas.')

    def predict(self, X):
        cent_preds = np.argmax(self.century_model.predict(X, verbose=0), axis=1)
        final = np.zeros(len(X), dtype=int)
        available = list(self.decade_submodels.keys())
        for c_idx in np.unique(cent_preds):
            c_use = c_idx if c_idx in available else min(available, key=lambda c: abs(c-c_idx))
            mask  = cent_preds == c_idx
            local = np.argmax(self.decade_submodels[c_use].predict(X[mask], verbose=0), axis=1)
            final[mask] = self.century_les[c_use].inverse_transform(local)
        return final

    def predict_proba(self, X):
        '''Probabilidades combinadas para ensemble.'''
        cent_probs = self.century_model.predict(X, verbose=0)  # (N, 4)
        final = np.zeros((len(X), NUM_CLASSES), dtype=np.float32)
        for c_idx, sub in self.decade_submodels.items():
            le_local = self.century_les[c_idx]
            dec_probs = sub.predict(X, verbose=0)  # (N, local_classes)
            c_weight  = cent_probs[:, c_idx]       # (N,)
            for li, gi in enumerate(le_local.classes_):
                final[:, gi] += c_weight * dec_probs[:, li]
        return final

print('HierarchicalModel definido.')


In [ ]:
hier = HierarchicalModel()
hier.fit(
    X_train_tfidf, y_train,
    X_val_tfidf,   y_val,
    century_train, century_val,
    batch_size=BATCH_SIZE, epochs=80
)

hier_val_preds  = hier.predict(X_val_tfidf)
hier_test_preds = hier.predict(X_test_tfidf)
hier_val  = accuracy_score(y_val,      hier_val_preds)
hier_test = accuracy_score(y_test_int, hier_test_preds)
print(f'Jerarquico — Val: {hier_val:.4f} | Test: {hier_test:.4f}')


## 11. BERT Español (fine-tuning)

Igual que en el intento 2. Solo se activa si PyTorch y Transformers estan instalados.


In [ ]:
if not BERT_AVAILABLE:
    print('BERT no disponible. Salta esta seccion.')
else:
    BERT_NAME    = 'dccuchile/bert-base-spanish-wwm-uncased'
    BERT_MAXLEN  = 512
    BERT_BATCH   = 16
    BERT_EPOCHS  = 5
    BERT_LR      = 3e-5
    BERT_OUTDIR  = PROCESS / 'bert' / 'best_model'

    class HistoricalDataset(Dataset):
        def __init__(self, texts, labels, tokenizer):
            self.texts     = texts
            self.labels    = labels
            self.tokenizer = tokenizer
        def __len__(self): return len(self.texts)
        def __getitem__(self, idx):
            enc = self.tokenizer(
                str(self.texts[idx]), max_length=BERT_MAXLEN,
                padding='max_length', truncation=True, return_tensors='pt')
            return {
                'input_ids':      enc['input_ids'].squeeze(0),
                'attention_mask': enc['attention_mask'].squeeze(0),
                'labels':         torch.tensor(self.labels[idx], dtype=torch.long)
            }

    bert_tokenizer = AutoTokenizer.from_pretrained(BERT_NAME)
    bert_model     = AutoModelForSequenceClassification.from_pretrained(
        BERT_NAME, num_labels=NUM_CLASSES, ignore_mismatched_sizes=True).to(DEVICE)

    # Congelar primeras 8 capas
    for name, param in bert_model.named_parameters():
        if any(f'layer.{i}.' in name for i in range(8)):
            param.requires_grad = False

    trainable = sum(p.numel() for p in bert_model.parameters() if p.requires_grad)
    total     = sum(p.numel() for p in bert_model.parameters())
    print(f'BERT params: {trainable:,} entrenables / {total:,} totales')

    ds_train = HistoricalDataset(X_train, y_train, bert_tokenizer)
    ds_val   = HistoricalDataset(X_val,   y_val,   bert_tokenizer)
    dl_train = DataLoader(ds_train, batch_size=BERT_BATCH, shuffle=True)
    dl_val   = DataLoader(ds_val,   batch_size=BERT_BATCH)

    no_decay = ['bias', 'LayerNorm.weight']
    params   = [
        {'params': [p for n,p in bert_model.named_parameters() if p.requires_grad and not any(nd in n for nd in no_decay)], 'weight_decay': 0.01},
        {'params': [p for n,p in bert_model.named_parameters() if p.requires_grad and     any(nd in n for nd in no_decay)], 'weight_decay': 0.0},
    ]
    optimizer = torch.optim.AdamW(params, lr=BERT_LR)
    total_steps = len(dl_train) * BERT_EPOCHS
    scheduler   = get_linear_schedule_with_warmup(
        optimizer, num_warmup_steps=int(0.1*total_steps), num_training_steps=total_steps)

    best_bert_acc = 0.0
    for epoch in range(BERT_EPOCHS):
        bert_model.train()
        total_loss = 0
        for batch in dl_train:
            optimizer.zero_grad()
            out  = bert_model(input_ids=batch['input_ids'].to(DEVICE),
                              attention_mask=batch['attention_mask'].to(DEVICE),
                              labels=batch['labels'].to(DEVICE))
            out.loss.backward()
            torch.nn.utils.clip_grad_norm_(bert_model.parameters(), 1.0)
            optimizer.step(); scheduler.step()
            total_loss += out.loss.item()

        bert_model.eval()
        all_preds, all_labels = [], []
        with torch.no_grad():
            for batch in dl_val:
                logits = bert_model(input_ids=batch['input_ids'].to(DEVICE),
                                    attention_mask=batch['attention_mask'].to(DEVICE)).logits
                all_preds.extend(torch.argmax(logits, dim=1).cpu().numpy())
                all_labels.extend(batch['labels'].numpy())

        val_acc = accuracy_score(all_labels, all_preds)
        print(f'BERT Epoch {epoch+1}/{BERT_EPOCHS} | Loss: {total_loss/len(dl_train):.4f} | Val Acc: {val_acc:.4f}')
        if val_acc > best_bert_acc:
            best_bert_acc = val_acc
            bert_model.save_pretrained(str(BERT_OUTDIR))
            bert_tokenizer.save_pretrained(str(BERT_OUTDIR))
            print(f'  -> Mejor BERT guardado (val_acc={best_bert_acc:.4f})')

        if MANUAL_STOP.get('stop', False):
            print('[PARADA MANUAL] BERT detenido.')
            MANUAL_STOP['stop'] = False
            break

    bert_val = best_bert_acc
    print(f'BERT — Mejor Val: {bert_val:.4f}')


In [ ]:
if BERT_AVAILABLE:
    def get_bert_probs(texts, tokenizer, model, batch_size=16):
        model.eval()
        probs_list = []
        for i in range(0, len(texts), batch_size):
            batch_texts = [str(t) for t in texts[i:i+batch_size]]
            enc = tokenizer(batch_texts, max_length=BERT_MAXLEN,
                            padding=True, truncation=True, return_tensors='pt')
            with torch.no_grad():
                logits = model(
                    input_ids=enc['input_ids'].to(DEVICE),
                    attention_mask=enc['attention_mask'].to(DEVICE)
                ).logits
            probs_list.append(torch.softmax(logits, dim=-1).cpu().numpy())
        return np.vstack(probs_list)

    bert_val_probs  = get_bert_probs(X_val,     bert_tokenizer, bert_model)
    bert_test_probs = get_bert_probs(X_test_int, bert_tokenizer, bert_model)
    bert_eval_probs = get_bert_probs(df_eval['text_clean'].astype(str).values, bert_tokenizer, bert_model)
    bert_test_acc   = accuracy_score(y_test_int, np.argmax(bert_test_probs, 1))
    print(f'BERT Test interno: {bert_test_acc:.4f}')
else:
    bert_val  = 0.0
    bert_test_probs = None


## 12. Ensamble ponderado por accuracy de validacion

In [ ]:
# Recolectar predicciones y probabilidades de todos los modelos
models_info = {
    'MLP_v2':       {'val': mlp_val,     'probs': model_mlp.predict(X_val_tfidf,    verbose=0),
                     'test':model_mlp.predict(X_test_tfidf,   verbose=0),
                     'eval':model_mlp.predict(X_eval_tfidf,   verbose=0)},
    'CNN_v2':       {'val': cnn_val,     'probs': model_cnn.predict(X_val_seq,      verbose=0),
                     'test':model_cnn.predict(X_test_seq,     verbose=0),
                     'eval':model_cnn.predict(X_eval_seq,     verbose=0)},
    'BiLSTM_v2':    {'val': bilstm_val,  'probs': model_bilstm.predict(X_val_seq,   verbose=0),
                     'test':model_bilstm.predict(X_test_seq,  verbose=0),
                     'eval':model_bilstm.predict(X_eval_seq,  verbose=0)},
    'CharCNN':      {'val': charcnn_val, 'probs': model_charcnn.predict(X_val_char, verbose=0),
                     'test':model_charcnn.predict(X_test_char,verbose=0),
                     'eval':model_charcnn.predict(X_eval_char,verbose=0)},
    'MLP_CharNgram':{'val': mlpchar_val, 'probs': model_mlpchar.predict(X_val_ctfidf, verbose=0),
                     'test':model_mlpchar.predict(X_test_ctfidf,verbose=0),
                     'eval':model_mlpchar.predict(X_eval_ctfidf,verbose=0)},
    'Transformer':  {'val': transf_val,  'probs': model_transf.predict(X_val_seq,   verbose=0),
                     'test':model_transf.predict(X_test_seq,  verbose=0),
                     'eval':model_transf.predict(X_eval_seq,  verbose=0)},
    'Jerarquico':   {'val': hier_val,    'probs': hier.predict_proba(X_val_tfidf),
                     'test':hier.predict_proba(X_test_tfidf),
                     'eval':hier.predict_proba(X_eval_tfidf)},
}

if BERT_AVAILABLE and bert_test_probs is not None:
    models_info['BERT'] = {
        'val': bert_val, 'probs': bert_val_probs,
        'test': bert_test_probs, 'eval': bert_eval_probs
    }

THRESHOLD = 0.05  # Solo incluir modelos que superen este accuracy

included = {n: d for n, d in models_info.items() if d['val'] >= THRESHOLD}
total_w  = sum(d['val'] for d in included.values())
weights  = {n: d['val']/total_w for n, d in included.items()}

print('Modelos incluidos en ensemble:')
for n, w in weights.items():
    print(f'  {n}: val={models_info[n]["val"]:.4f} | peso={w:.3f}')

ens_val_probs  = sum(w * included[n]['probs'] for n, w in weights.items())
ens_test_probs = sum(w * included[n]['test']  for n, w in weights.items())
ens_eval_probs = sum(w * included[n]['eval']  for n, w in weights.items())

ens_val  = accuracy_score(y_val,      np.argmax(ens_val_probs,  1))
ens_test = accuracy_score(y_test_int, np.argmax(ens_test_probs, 1))
print(f'\nENSAMBLE — Val: {ens_val:.4f} | Test interno: {ens_test:.4f}')


## 13. Comparacion de modelos

In [ ]:
all_results = {n: {'val': d['val'], 'test': accuracy_score(y_test_int, np.argmax(d['test'], 1))}
               for n, d in models_info.items()}
all_results['ENSEMBLE'] = {'val': ens_val, 'test': ens_test}

df_res = pd.DataFrame(all_results).T.sort_values('val', ascending=False)
print(df_res.to_string())

# Grafica comparativa
fig, ax = plt.subplots(figsize=(12, 5))
x = range(len(df_res))
ax.bar([i-0.2 for i in x], df_res['val'],  width=0.4, label='Val',           color='steelblue', alpha=0.8)
ax.bar([i+0.2 for i in x], df_res['test'], width=0.4, label='Test interno',  color='coral',     alpha=0.8)
ax.set_xticks(list(x)); ax.set_xticklabels(df_res.index, rotation=45, ha='right')
ax.set_ylabel('Accuracy'); ax.set_title('Comparacion de modelos — Intento 3')
ax.legend(); plt.tight_layout()
plt.savefig(ipath('comparacion_modelos_intento3.png'), dpi=100, bbox_inches='tight')
plt.show()

best_model_name = df_res.index[0]
print(f'Mejor modelo: {best_model_name} (val={df_res.loc[best_model_name, "val"]:.4f})')


## 14. Generacion de submission

In [ ]:
# Usar ensemble para la prediccion final en eval
kaggle_preds = label_encoder.inverse_transform(np.argmax(ens_eval_probs, axis=1))

submission = pd.DataFrame({'id': df_eval['id'], 'answer': kaggle_preds})
sub_path   = f'submission_ensemble_intento3.csv'
submission.to_csv(sub_path, index=False)

print(f'Submission guardado: {sub_path}')
print(f'Filas: {len(submission)}')
print(f'Decadas predichas (muestra): {sorted(submission["answer"].unique())[:10]}')
print(submission.head())


In [ ]:
# Guardar metricas finales
metrics = {
    'modelos': {n: {'val': float(d['val']), 'test': float(all_results[n]['test'])}
                for n, d in models_info.items()},
    'ensemble': {'val': float(ens_val), 'test': float(ens_test)},
    'num_classes': int(NUM_CLASSES),
    'total_train': int(len(X_train)),
}
with open(str(PROCESS / 'metrics_intento3.json'), 'w') as f:
    json.dump(metrics, f, indent=2)
print('Metricas guardadas en process/metrics_intento3.json')
